# 🚀 Notebook do Professor (Demo) — Aula 07: RAG avançado — chunking estratégico, reranking e RAGAS

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 07/14 — Módulo 2: RAG · 🏁 Entrega CKP02**  
**⏱️ 1h40min**  
**📊 faithfulness · answer_relevancy**  
**🏁 CKP02 entrega**  

---

## 🎯 Objetivo da aula

Sair da fase "funciona" para a fase "funciona bem". Medir objetivamente a qualidade do RAG com RAGAS, experimentar duas estratégias de chunking e documentar qual configuração entrega melhores resultados para o domínio do grupo. Isso é o CKP02.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 06 — Três estratégias de chunking — quando usar cada uma

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

# SemanticChunker usa o modelo de embedding para decidir onde quebrar
semantic_splitter = SemanticChunker(
    embeddings,                         # mesmo OllamaEmbeddings da Aula 06
    breakpoint_threshold_type="percentile",  # quebra nos 95% de maior divergência
)
chunks_sem = semantic_splitter.split_documents(paginas)
print(f"Chunks semânticos: {len(chunks_sem)}")
print(f"Tamanho médio: {sum(len(c.page_content) for c in chunks_sem)/len(chunks_sem):.0f} chars")

### Slide 07 — Parent-Document Retriever — o melhor dos dois mundos

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

# Splitter pequeno — para indexação e busca (precisão)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# Splitter grande — para o contexto enviado ao LLM (riqueza)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)

# Armazenamento para os chunks pai (não vai no ChromaDB)
store = InMemoryStore()

retriever_pd = ParentDocumentRetriever(
    vectorstore=db,             # ChromaDB com chunks de 200 chars
    docstore=store,            # store com chunks de 1000 chars
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
retriever_pd.add_documents(paginas)

# Como funciona:
# 1. Query → ChromaDB busca nos chunks de 200 chars (alta precisão)
# 2. Encontra chunk filho → recupera o chunk PAI de 1000 chars do store
# 3. Retorna o chunk de 1000 chars para o LLM (contexto rico)
docs = retriever_pd.invoke("prazo de garantia")
print(f"Docs retornados: {len(docs)}")
print(f"Tamanho do 1º: {len(docs[0].page_content)} chars")  # → ~1000

### Slide 09 — Reranking — o segundo filtro de relevância

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

# cross-encoder leve, gratuito (roda em CPU no Colab)
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
compressor    = CrossEncoderReranker(model=cross_encoder, top_n=3)

# Combina retriever (busca ampla, k=10) + reranker (filtra para top_n=3)
retriever_rerank = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=db.as_retriever(search_kwargs={"k":10}),  # busca ampla
)
# Trocar o retriever na chain RAG é tudo que precisa mudar

### Slide 12 — RAGAS — implementação simplificada para o CKP02

In [ ]:
!pip install ragas -q

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# Configurar o RAGAS para usar o Ollama (sem custo)
ragas_llm  = LangchainLLMWrapper(ChatOllama(model="gpt-oss:120b", temperature=0))
ragas_embs = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))

faithfulness.llm       = ragas_llm
answer_relevancy.llm   = ragas_llm
answer_relevancy.embeddings = ragas_embs

# Dataset de avaliação — 5 pares preparados antes da aula
dados_avaliacao = {
    "question": [
        "Qual é o prazo de garantia do produto?",
        "Como acionar o suporte técnico?",
    ],
    "answer": [
        chain_rag.invoke("Qual é o prazo de garantia do produto?"),
        chain_rag.invoke("Como acionar o suporte técnico?"),
    ],
    "contexts": [
        [d.page_content for d in retriever.invoke("garantia")],
        [d.page_content for d in retriever.invoke("suporte técnico")],
    ],
}

resultado = evaluate(
    Dataset.from_dict(dados_avaliacao),
    metrics=[faithfulness, answer_relevancy],
)
print(resultado.to_pandas()[["question", "faithfulness", "answer_relevancy"]])

### Slide 14 — Fallback — RAGAS manual se o pacote tiver problema de compatibilidade

In [ ]:
# Implementação manual de faithfulness (LLM-as-judge)
PROMPT_JUIZ = """<tarefa>
Você é um avaliador de qualidade de sistemas RAG.
Avalie se a RESPOSTA está fundamentada no CONTEXTO.
</tarefa>

<contexto>{contexto}</contexto>
<resposta>{resposta}</resposta>

Responda APENAS com um número de 0 a 1:
- 1.0: toda a resposta está no contexto
- 0.5: resposta parcialmente no contexto
- 0.0: resposta inventa informações não presentes no contexto

Resposta (só o número):"""

def faithfulness_manual(pergunta, resposta, contexto) -> float:
    juiz = ChatOllama(model="gpt-oss:120b", temperature=0)
    prompt = ChatPromptTemplate.from_template(PROMPT_JUIZ)
    chain_juiz = prompt | juiz | StrOutputParser()
    resultado = chain_juiz.invoke({
        "contexto": contexto, "resposta": resposta
    })
    try:
        return float(resultado.strip())
    except:
        return -1.0  # score inválido — revisar resposta do modelo

# Aplicar a todas as perguntas
scores = []
for q in PERGUNTAS:
    resp = chain_rag.invoke(q)
    ctx  = "\n".join(d.page_content for d in retriever.invoke(q))
    scores.append(faithfulness_manual(q, resp, ctx))
print(f"Faithfulness médio: {sum(scores)/len(scores):.3f}")

### Slide 16 — Código da demo — comparar chunk_size com RAGAS

In [ ]:
def avaliar_chunking(paginas, chunk_size: int, perguntas: list) -> dict:
    """Cria pipeline RAG com chunk_size dado e retorna scores RAGAS."""
    chunks    = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_size//8).split_documents(paginas)
    retriever = Chroma.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k":3})
    chain     = montar_chain_rag(retriever)

    dados = {"question":[], "answer":[], "contexts":[]}
    for q in perguntas:
        dados["question"].append(q)
        dados["answer"].append(chain.invoke(q))
        dados["contexts"].append([d.page_content for d in retriever.invoke(q)])

    res = evaluate(Dataset.from_dict(dados), metrics=[faithfulness, answer_relevancy])
    return {"chunk_size": chunk_size,
            "faithfulness":     res["faithfulness"],
            "answer_relevancy": res["answer_relevancy"]}

for cs in [256, 512, 1024]:
    r = avaliar_chunking(paginas, cs, PERGUNTAS_AVALIACAO)
    print(f"chunk={r['chunk_size']:4d} | faith={r['faithfulness']:.3f} | rel={r['answer_relevancy']:.3f}")

### Slide 17 — Python novo desta aula

In [ ]:
# 1. Divisão inteira com // — chunk_overlap como fração do chunk_size
chunk_size    = 512
chunk_overlap = chunk_size // 8  # → 64 (divisão inteira, sem float)

# 2. Loop com tupla de 3 elementos — (nome, chain, retriever)
for nome, chain, retr in [
    ("A", chain_a, retriever_a),
    ("B", chain_b, retriever_b),
]:
    print(f"Testando {nome}")

# 3. try / except com ValueError para float parsing
try:
    score = float(texto.strip())  # LLM pode retornar "0.87" ou texto extra
except ValueError:
    score = -1.0               # sentinela: score inválido

# 4. List comprehension aninhado — contexts para o RAGAS
contexts = [
    [d.page_content for d in retriever.invoke(q)]
    for q in perguntas
]  # lista de listas: [["chunk1", "chunk2", ...], [...], ...]

# 5. sum() / len() para média simples
media = sum(scores) / len(scores)  # sem precisar importar statistics

# 6. Dataset.from_dict() — criar dataset HuggingFace de um dict
from datasets import Dataset
ds = Dataset.from_dict({"col_a": [1,2], "col_b": ["x","y"]})

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 07 · CKP02 R2 e R3**  
### Otimizar RAG com RAGAS — 2 estratégias de chunking ★★★

*Grupo 3–4 · 20 minutos · Google Colab · PDFs do domínio obrigatórios*

1. Complete as 4 lacunas — carregar PDFs, instanciar SemanticChunker, montar retriever_b e preencher PERGUNTAS com 5 perguntas reais do domínio.
2. Compare as duas estratégias — documente em uma célula markdown qual obteve melhores scores de faithfulness e answer_relevancy e por que (hipótese do grupo).
3. Análise de falhas: identifique 1 pergunta onde a chain errou (faithfulness baixo) — recupere os chunks que foram usados e explique por que o retriever não trouxe o trecho certo.

> **🎯 Gabarito das lacunas**
>
> Lacuna 1: PyMuPDFLoader(pdf).load()
>
> Lacuna 2: SemanticChunker(embeddings)
>
> Lacuna 3: db_b.as_retriever(search_kwargs={"k": 3})
>
> Lacuna 4: PERGUNTAS (no loop de contexts)

In [ ]:
!pip install langchain langchain-community langchain-ollama pymupdf chromadb ragas datasets langchain-experimental langchain-text-splitters -q

# Setup (idêntico à Aula 06)
import os
from google.colab import userdata, files
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# 👉 LACUNA 1: carregue os PDFs do grupo (mesmos da Aula 06)
paginas = []
for pdf in pdf_paths:
    paginas.extend(PyMuPDFLoader(___).load())

# ── ESTRATÉGIA A: Recursive com chunk_size=512 (vs. 800 da Aula 06) ──
chunks_a = RecursiveCharacterTextSplitter(
    chunk_size=___, chunk_overlap=___
).split_documents(paginas)
db_a         = Chroma.from_documents(chunks_a, embeddings, persist_directory="/content/ckp02_a")
retriever_a  = db_a.as_retriever(search_kwargs={"k":3})
chain_a      = montar_chain_rag(retriever_a)

# ── ESTRATÉGIA B: SemanticChunker ────────────────────────────────────
# 👉 LACUNA 2: instancie o SemanticChunker com o modelo de embedding
chunks_b    = SemanticChunker(___).split_documents(paginas)
db_b        = Chroma.from_documents(chunks_b, embeddings, persist_directory="/content/ckp02_b")
retriever_b = ___  # retriever com k=3
chain_b     = montar_chain_rag(retriever_b)

# 👉 LACUNA 3: defina 5 perguntas reais do domínio para avaliação
PERGUNTAS = [___, ___, ___, ___, ___]

# 👉 LACUNA 4: monte o dataset RAGAS e avalie as duas estratégias
for nome, chain, retr in [
    ("Recursive-512", chain_a, retriever_a),
    ("Semantic",      chain_b, retriever_b),
]:
    dados = {
        "question": PERGUNTAS,
        "answer":   [chain.invoke(q) for q in PERGUNTAS],
        "contexts": [[d.page_content for d in retr.invoke(q)] for q in ___],
    }
    res = evaluate(Dataset.from_dict(dados), metrics=[faithfulness, answer_relevancy])
    print(f"{nome}: faith={res['faithfulness']:.3f} · rel={res['answer_relevancy']:.3f}")

## 📚 Referências da aula

- Paper Es, S. et al. — "RAGAS: Automated Evaluation of Retrieval Augmented Generation." EACL, 2024. O paper que define faithfulness e answer_relevancy. arxiv.org/abs/2309.15217
- Docs RAGAS — Documentação oficial: métricas, integração com Ollama, datasets. docs.ragas.io
- Docs LangChain — SemanticChunker e ParentDocumentRetriever. python.langchain.com/docs/how_to/semantic-chunker
- Modelo cross-encoder/ms-marco-MiniLM-L-6-v2 — Modelo de reranking leve (22M params). huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings que sustentam o chunking semântico.

---

**→ Próxima Aula — Aula 08 · 29/Set** — Interfaces com Gradio e Streamlit
  
RAG com URL pública. RunnableWithMessageHistory para memória entre turnos.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*